# 🛒 Retrain Model Prediksi Tren Pasar UMKM Bogor — v3

**Notebook ini melatih ulang model Random Forest + Logistic Regression untuk memprediksi
 daya tarik produk UMKM Bogor.**

### Fitur utama v3
| Fitur | Keterangan |
|---|---|
| `rasio_harga` | Harga produk ÷ median harga kategori (konteks pasar) |
| `zscore_harga` | Seberapa jauh harga dari rata-rata kategori |
| `log_harga` | Log-transform untuk stabilkan distribusi skewed |
| `segmen_harga` | Murah / Menengah / Mahal berdasarkan kuartil per kategori |
| `rating` | Rating produk (nilai 0 diganti rata-rata non-zero) |

### Label
Label dibuat dari **skor daya tarik gabungan**:
```
skor = 0.7 × norm(jumlah_terjual) + 0.3 × norm(rating)
label = 1  jika skor > median(skor)   → Menarik
label = 0  jika skor ≤ median(skor)   → Kurang Menarik
```

### Output file yang dihasilkan
- `model_umkm_bogor_v2.joblib` — model RF final
- `market_stats_per_kategori.csv` — statistik harga pasar
- `tren_per_kategori.csv` — peringkat tren per kategori
- `tren_per_sub_kategori.csv` — peringkat tren per sub-kategori
- `confusion_matrix_comparison.png` — confusion matrix RF vs LR

---
## ⚙️ Setup — Install & Upload

> Jalankan cell ini **pertama kali** sebelum yang lain.
> Setelah install selesai, upload file `dataset_umkm_bogor.csv`.

In [ ]:
# Install library yang tidak tersedia by default di Colab
!pip install PySastrawi==1.0.1 --quiet
print('✅ PySastrawi terinstall')

In [ ]:
# Upload dataset ke Colab runtime
from google.colab import files
print('Silakan pilih file: dataset_umkm_bogor.csv')
uploaded = files.upload()
print('\n✅ File terupload:', list(uploaded.keys()))

---
## 📦 Import Libraries

In [ ]:
import re
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
print('✅ Semua library berhasil diimport')

---
## 📂 Step 1 — Load Dataset

In [ ]:
df = pd.read_csv('dataset_umkm_bogor.csv')
print(f'✅ Dataset dimuat: {len(df)} baris, {len(df.columns)} kolom')
print(f'   Kolom: {list(df.columns)}')
df.head()

---
## 🧹 Step 2 — Bersihkan Nilai 0 pada `jumlah_terjual` & `rating`

Nilai 0 pada kedua kolom ini menyebabkan distribusi data sangat condong ke nol
sehingga membiaskan model. Solusi: **ganti nilai 0 dengan rata-rata non-zero** per kolom.

- `jumlah_terjual = 0` → produk tidak pernah terjual, bukan nilai valid untuk training
- `rating = 0` → produk belum punya rating, bukan berarti rating-nya 0

In [ ]:
zero_terjual = (df['jumlah_terjual'] == 0).sum()
zero_rating  = (df['rating'] == 0).sum()

mean_terjual_nonzero = df.loc[df['jumlah_terjual'] > 0, 'jumlah_terjual'].mean()
mean_rating_nonzero  = df.loc[df['rating'] > 0, 'rating'].mean()

df['jumlah_terjual'] = df['jumlah_terjual'].replace(0, round(mean_terjual_nonzero, 4))
df['rating']         = df['rating'].replace(0, round(mean_rating_nonzero, 4))

print(f'jumlah_terjual — {zero_terjual} nilai 0 → diganti rata-rata non-zero: {mean_terjual_nonzero:.2f}')
print(f'rating         — {zero_rating}  nilai 0 → diganti rata-rata non-zero: {mean_rating_nonzero:.2f}')
print(f'Range rating setelah perbaikan: {df["rating"].min():.2f} – {df["rating"].max():.2f}')

# Visualisasi distribusi sebelum vs sesudah (rating)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['jumlah_terjual'].hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Distribusi jumlah_terjual (setelah clean)', fontweight='bold')
axes[0].set_xlabel('Jumlah Terjual')
axes[0].set_ylabel('Frekuensi')
df['rating'].hist(bins=30, ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Distribusi rating (setelah clean)', fontweight='bold')
axes[1].set_xlabel('Rating')
plt.tight_layout()
plt.show()

---
## ✏️ Step 3 — Text Preprocessing (Sastrawi)

Nama produk dibersihkan menggunakan stemmer bahasa Indonesia (PySastrawi):
1. Lowercase semua karakter
2. Hapus karakter non-alfabet
3. Hapus stopwords domain (murah, promo, cod, gram, dll.)
4. Stemming dengan Sastrawi

In [ ]:
stemmer   = StemmerFactory().create_stemmer()
stopwords = {
    'murah', 'promo', 'cod', 'terlaris', 'original', 'ori', 'asli',
    'oleh', 'pcs', 'gr', 'gram', 'kg', 'dan', 'di', 'ke', 'dari', 'yang'
}

def clean_text(text):
    text  = str(text).lower()
    text  = re.sub(r'[^a-z\s]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stopwords]
    return stemmer.stem(' '.join(words))

df['nama_produk_clean'] = df['nama_produk'].apply(clean_text)

print('✅ Text preprocessing selesai!')
print()
# Contoh hasil
sample = df[['nama_produk', 'nama_produk_clean']].head(5)
print(sample.to_string(index=False))

---
## 🔧 Step 4 — Feature Engineering Bisnis

Fitur harga **relatif terhadap konteks pasar** per kategori:

| Fitur | Formula | Makna Bisnis |
|---|---|---|
| `rasio_harga` | `harga / median_kategori` | 1.0 = di median, 2.0 = 2× lebih mahal |
| `zscore_harga` | `(harga - mean) / std` | Seberapa ekstrem harga vs rata-rata |
| `log_harga` | `log1p(harga)` | Stabilkan distribusi skewed |
| `segmen_harga` | Kuartil Q25/Q75 | 0=Murah, 1=Menengah, 2=Mahal |

> 💡 Harga Rp 1.000.000 untuk produk dengan median pasar Rp 50.000
> → `rasio_harga = 20×` → model tahu ini **sangat mahal** untuk kategorinya.

In [ ]:
# Statistik harga per kategori (konteks pasar)
stat_per_kategori = df.groupby('kategori')['harga_produk'].agg(
    median_harga_kategori='median',
    mean_harga_kategori='mean',
    std_harga_kategori='std',
    q25_harga_kategori=lambda x: x.quantile(0.25),
    q75_harga_kategori=lambda x: x.quantile(0.75),
).reset_index()

stat_per_kategori.to_csv('market_stats_per_kategori.csv', index=False)
print('✅ market_stats_per_kategori.csv disimpan')
print()
stat_per_kategori

In [ ]:
# Merge statistik ke dataset
df = df.merge(stat_per_kategori, on='kategori', how='left')

# Rasio harga vs median pasar
df['rasio_harga'] = df['harga_produk'] / df['median_harga_kategori'].replace(0, np.nan)
df['rasio_harga'] = df['rasio_harga'].fillna(1.0).clip(upper=50)

# Z-score harga dalam kategori
df['zscore_harga'] = (
    (df['harga_produk'] - df['mean_harga_kategori'])
    / df['std_harga_kategori'].replace(0, 1)
).clip(-5, 5)

# Log harga
df['log_harga'] = np.log1p(df['harga_produk'])

# Segmen harga per kategori (0=Murah, 1=Menengah, 2=Mahal)
def segment_harga(row):
    if row['harga_produk'] <= row['q25_harga_kategori']:
        return 0  # Murah
    elif row['harga_produk'] <= row['q75_harga_kategori']:
        return 1  # Menengah
    else:
        return 2  # Mahal

df['segmen_harga'] = df.apply(segment_harga, axis=1)

print('Statistik rasio_harga:')
print(f'  Min    : {df["rasio_harga"].min():.2f}x')
print(f'  Median : {df["rasio_harga"].median():.2f}x')
print(f'  Max    : {df["rasio_harga"].max():.2f}x')
print(f'  > 5x   : {(df["rasio_harga"] > 5).sum()} produk')
print()
print('Distribusi segmen harga:')
print(df['segmen_harga'].value_counts().rename({0:'Murah', 1:'Menengah', 2:'Mahal'}).to_string())

---
## 🏷️ Step 5 — Labeling — Skor Daya Tarik Gabungan

**Metode lama** yang buruk: `label = jumlah_terjual > 10` → rule angka tetap, tidak adil

**Metode baru**: skor gabungan yang mempertimbangkan **popularitas + kualitas**:

```
skor_daya_tarik = 0.7 × norm(jumlah_terjual) + 0.3 × norm(rating)
label = 1  jika skor > median(skor)   →  Menarik
label = 0  jika skor ≤ median(skor)   →  Kurang Menarik
```

Bobot **70:30** karena penjualan lebih langsung mencerminkan tren pasar.

In [ ]:
min_t = df['jumlah_terjual'].min()
max_t = df['jumlah_terjual'].max()
df['norm_terjual']    = (df['jumlah_terjual'] - min_t) / max(max_t - min_t, 1)
df['norm_rating']     = df['rating'] / 5.0
df['skor_daya_tarik'] = 0.7 * df['norm_terjual'] + 0.3 * df['norm_rating']

threshold_skor = df['skor_daya_tarik'].median()
df['label']    = (df['skor_daya_tarik'] > threshold_skor).astype(int)

print(f'Threshold skor (median) : {threshold_skor:.4f}')
print(f'Min skor                : {df["skor_daya_tarik"].min():.4f}')
print(f'Max skor                : {df["skor_daya_tarik"].max():.4f}')
print()
label_counts = df['label'].value_counts().rename({0: 'Kurang Menarik', 1: 'Menarik'})
print('Distribusi label:')
print(label_counts.to_string())
print(f'Rasio    : {label_counts["Menarik"] / len(df) * 100:.1f}% Menarik  |  '
      f'{label_counts["Kurang Menarik"] / len(df) * 100:.1f}% Kurang Menarik')

# Visualisasi distribusi skor
plt.figure(figsize=(8, 4))
plt.hist(df[df['label']==0]['skor_daya_tarik'], bins=40, alpha=0.7,
         color='salmon', label='Kurang Menarik')
plt.hist(df[df['label']==1]['skor_daya_tarik'], bins=40, alpha=0.7,
         color='steelblue', label='Menarik')
plt.axvline(threshold_skor, color='red', linestyle='--', linewidth=2,
            label=f'Threshold = {threshold_skor:.4f}')
plt.title('Distribusi Skor Daya Tarik per Label', fontweight='bold')
plt.xlabel('Skor Daya Tarik')
plt.ylabel('Frekuensi')
plt.legend()
plt.tight_layout()
plt.show()

---
## 📊 Step 6 — Ekspor Statistik Tren per Kategori & Sub-Kategori

File CSV ini dipakai oleh `app.py` untuk menampilkan blok `tren_keseluruhan`
di setiap response prediksi — menjawab pertanyaan: *"Dari semua kategori,
mana yang paling digemari pembeli secara keseluruhan?"*

In [ ]:
# ── Tren per KATEGORI ──────────────────────────────────────────────────
tren_per_kategori = (
    df.groupby('kategori')
    .agg(
        rata_rata_terjual=('jumlah_terjual', 'mean'),
        median_terjual=('jumlah_terjual', 'median'),
        rata_rata_rating=('rating', 'mean'),
        jumlah_produk=('nama_produk', 'count'),
        rata_rata_skor=('skor_daya_tarik', 'mean'),
    )
    .reset_index()
)

max_avg_terjual = tren_per_kategori['rata_rata_terjual'].max()
tren_per_kategori['skor_tren'] = (
    0.7 * (tren_per_kategori['rata_rata_terjual'] / max(max_avg_terjual, 1))
    + 0.3 * (tren_per_kategori['rata_rata_rating'] / 5.0)
)
tren_per_kategori = tren_per_kategori.sort_values(
    by='skor_tren', ascending=False
).reset_index(drop=True)
tren_per_kategori['peringkat'] = range(1, len(tren_per_kategori) + 1)

tren_per_kategori.to_csv('tren_per_kategori.csv', index=False)
print('✅ tren_per_kategori.csv disimpan')
print()
tren_per_kategori[['peringkat', 'kategori', 'rata_rata_terjual',
                    'rata_rata_rating', 'jumlah_produk', 'skor_tren']]

In [ ]:
# ── Tren per SUB-KATEGORI ───────────────────────────────────────────────
tren_per_sub = (
    df.groupby(['kategori', 'sub_kategori'])
    .agg(
        rata_rata_terjual=('jumlah_terjual', 'mean'),
        median_terjual=('jumlah_terjual', 'median'),
        rata_rata_rating=('rating', 'mean'),
        jumlah_produk=('nama_produk', 'count'),
    )
    .reset_index()
)

# Gunakan transform() agar kolom 'kategori' tidak hilang ke index (pandas 2.x)
max_terjual_per_kat = (
    tren_per_sub.groupby('kategori')['rata_rata_terjual']
    .transform('max')
    .clip(lower=1)
)
tren_per_sub['skor_tren'] = (
    0.7 * (tren_per_sub['rata_rata_terjual'] / max_terjual_per_kat)
    + 0.3 * (tren_per_sub['rata_rata_rating'] / 5.0)
)
tren_per_sub = tren_per_sub.sort_values(
    ['kategori', 'skor_tren'], ascending=[True, False]
)

tren_per_sub.to_csv('tren_per_sub_kategori.csv', index=False)
print('✅ tren_per_sub_kategori.csv disimpan')
print()
tren_per_sub.head(10)

---
## 🗂️ Step 7 — Feature Matrix & Train-Test Split

Fitur yang digunakan model:
- **Text**: `nama_produk_clean` → TF-IDF (1000 fitur, unigram + bigram)
- **Kategorikal**: `kategori`, `sub_kategori` → One-Hot Encoding
- **Numerik**: `rasio_harga`, `zscore_harga`, `log_harga`, `segmen_harga`, `rating` → StandardScaler

In [ ]:
text_feature = 'nama_produk_clean'
cat_features = ['kategori', 'sub_kategori']
num_features = [
    'rasio_harga',   # harga relatif terhadap pasar per kategori
    'zscore_harga',  # seberapa jauh dari rata-rata pasar
    'log_harga',     # stabilkan distribusi harga skewed
    'segmen_harga',  # murah / menengah / mahal
    'rating',        # kualitas produk (sudah bersih dari nilai 0)
]

X = df[[text_feature] + cat_features + num_features]
y = df['label']

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Total data     : {len(X)}')
print(f'Train+Val set  : {len(X_train_val)} ({len(X_train_val)/len(X)*100:.0f}%)')
print(f'Test set       : {len(X_test)} ({len(X_test)/len(X)*100:.0f}%)')
print()
print('Distribusi label di test set:')
print(y_test.value_counts().rename({0:'Kurang Menarik', 1:'Menarik'}).to_string())

---
## 🔩 Step 8 — Pipeline Preprocessing

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(max_features=1000, ngram_range=(1, 2)), text_feature),
        ('cat',  OneHotEncoder(handle_unknown='ignore'), cat_features),
        ('num',  StandardScaler(), num_features),
    ]
)

print('✅ Preprocessor pipeline siap')
print('   - TF-IDF     : max_features=1000, ngram=(1,2)')
print('   - OneHotEnc  : handle_unknown=ignore')
print('   - StandardScaler: rasio_harga, zscore_harga, log_harga, segmen_harga, rating')

---
## 🔄 Step 8 & 8b — Cross Validation: Random Forest vs Logistic Regression

**5-Fold Stratified CV** dijalankan untuk **dua model** secara berurutan:
1. **Random Forest** (baseline 100 trees)
2. **Logistic Regression** (pembanding, `max_iter=1000`)

Hasilnya dibandingkan langsung dalam satu tabel.

In [ ]:
skf     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

# ── Random Forest baseline ──────────────────────────────────────────────
rf_baseline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

print('Menjalankan CV Random Forest...')
cv_results = cross_validate(
    rf_baseline, X_train_val, y_train_val,
    cv=skf, scoring=scoring, return_train_score=False
)

# ── Logistic Regression baseline ────────────────────────────────────────
lr_baseline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        max_iter=1000, solver='lbfgs',
        class_weight='balanced', random_state=42
    ))
])

print('Menjalankan CV Logistic Regression...')
cv_results_lr = cross_validate(
    lr_baseline, X_train_val, y_train_val,
    cv=skf, scoring=scoring, return_train_score=False
)

# ── Tabel perbandingan CV ────────────────────────────────────────────────
print()
print(f"{'Metrik':<15} {'RF Mean':>10} {'RF Std':>8}  {'LR Mean':>10} {'LR Std':>8}  Unggul")
print('-' * 70)
for metric in scoring:
    rf_s = cv_results[f'test_{metric}']
    lr_s = cv_results_lr[f'test_{metric}']
    winner = 'RF ✅' if rf_s.mean() >= lr_s.mean() else 'LR ✅'
    print(f"{metric:<15} {rf_s.mean():>10.4f} {rf_s.std():>8.4f}"
          f"  {lr_s.mean():>10.4f} {lr_s.std():>8.4f}  {winner}")

---
## ⚡ Step 9 — Hyperparameter Tuning Random Forest (GridSearchCV)

> ⏱️ **Estimasi waktu: 15–40 menit** tergantung ukuran dataset.
> GridSearchCV menguji **72 kombinasi parameter** × 5-fold CV.

Parameter yang di-tune:
- `n_estimators`: jumlah pohon [100, 200, 300]
- `max_depth`: kedalaman pohon [None, 10, 20]
- `min_samples_split`: [2, 5]
- `min_samples_leaf`: [1, 2]
- `class_weight`: ['balanced', None]

In [ ]:
param_grid = {
    'classifier__n_estimators'   : [100, 200, 300],
    'classifier__max_depth'      : [None, 10, 20],
    'classifier__min_samples_split': [2, 5],
    'classifier__min_samples_leaf' : [1, 2],
    'classifier__class_weight'   : ['balanced', None],
}

rf_for_grid = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

grid_search = GridSearchCV(
    rf_for_grid, param_grid, cv=skf,
    scoring='roc_auc',
    n_jobs=-1,   # pakai semua core Colab
    verbose=1
)
grid_search.fit(X_train_val, y_train_val)

print('\n✅ GridSearchCV selesai!')
print(f'Best Parameters : {grid_search.best_params_}')
print(f'Best AUC-ROC CV : {grid_search.best_score_:.4f}')

---
## 📈 Step 10 — Evaluasi di Holdout Test Set

Evaluasi **Random Forest** (model terbaik dari GridSearchCV) pada data test
yang belum pernah dilihat model selama training.

In [ ]:
best_rf = grid_search.best_estimator_
y_pred  = best_rf.predict(X_test)
y_proba = best_rf.predict_proba(X_test)[:, 1]

print('── Random Forest (Tuned) ──────────────────────────────────')
print(f"{'Accuracy':<15} {accuracy_score(y_test, y_pred):.4f}")
print(f"{'Precision':<15} {precision_score(y_test, y_pred):.4f}")
print(f"{'Recall':<15} {recall_score(y_test, y_pred):.4f}")
print(f"{'F1 Score':<15} {f1_score(y_test, y_pred):.4f}")
print(f"{'AUC-ROC':<15} {roc_auc_score(y_test, y_proba):.4f}")
print()
print(classification_report(y_test, y_pred, target_names=['Kurang Menarik', 'Menarik']))

---
## 📉 Step 10b — Evaluasi Logistic Regression di Test Set

Logistic Regression dilatih di train set (tanpa hyperparameter tuning)
dan dievaluasi pada test set yang sama — sebagai **baseline pembanding**.

In [ ]:
lr_baseline.fit(X_train_val, y_train_val)
y_pred_lr  = lr_baseline.predict(X_test)
y_proba_lr = lr_baseline.predict_proba(X_test)[:, 1]

print('── Logistic Regression (Baseline) ─────────────────────────')
print(f"{'Accuracy':<15} {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"{'Precision':<15} {precision_score(y_test, y_pred_lr):.4f}")
print(f"{'Recall':<15} {recall_score(y_test, y_pred_lr):.4f}")
print(f"{'F1 Score':<15} {f1_score(y_test, y_pred_lr):.4f}")
print(f"{'AUC-ROC':<15} {roc_auc_score(y_test, y_proba_lr):.4f}")
print()
print(classification_report(y_test, y_pred_lr, target_names=['Kurang Menarik', 'Menarik']))

---
## 🏆 Tabel Perbandingan & Confusion Matrix

Perbandingan langsung **Random Forest (Tuned)** vs **Logistic Regression**
pada semua metrik evaluasi, diikuti Confusion Matrix side-by-side.

In [ ]:
metrics_compare = {
    'Accuracy' : (accuracy_score(y_test, y_pred),    accuracy_score(y_test, y_pred_lr)),
    'Precision': (precision_score(y_test, y_pred),   precision_score(y_test, y_pred_lr)),
    'Recall'   : (recall_score(y_test, y_pred),      recall_score(y_test, y_pred_lr)),
    'F1 Score' : (f1_score(y_test, y_pred),          f1_score(y_test, y_pred_lr)),
    'AUC-ROC'  : (roc_auc_score(y_test, y_proba),    roc_auc_score(y_test, y_proba_lr)),
}

print('=' * 60)
print('  PERBANDINGAN: Random Forest (Tuned) vs Logistic Regression')
print('=' * 60)
print(f"{'Metrik':<12} {'Random Forest':>15} {'Logistic Reg':>14} {'Unggul':>8}")
print('-' * 55)
rf_wins = lr_wins = 0
for name, (rf_val, lr_val) in metrics_compare.items():
    if rf_val >= lr_val:
        winner = 'RF ✅'; rf_wins += 1
    else:
        winner = 'LR ✅'; lr_wins += 1
    print(f"{name:<12} {rf_val:>15.4f} {lr_val:>14.4f} {winner:>8}")
print('-' * 55)
print(f"{'Total Wins':<12} {'RF: ' + str(rf_wins):>15} {'LR: ' + str(lr_wins):>14}")
print()
if rf_wins > lr_wins:
    print('→ Random Forest lebih unggul: dipilih sebagai model produksi ✅')
else:
    print('→ Logistic Regression lebih unggul atau seimbang — pertimbangkan LR')

In [ ]:
# ── Confusion Matrix side-by-side ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Confusion Matrix: Random Forest vs Logistic Regression',
             fontsize=14, fontweight='bold', y=1.02)

models_info = [
    (y_pred,    'Random Forest (Tuned)',          'Blues'),
    (y_pred_lr, 'Logistic Regression (Baseline)', 'Oranges'),
]

for ax, (y_p, title, cmap) in zip(axes, models_info):
    cm   = confusion_matrix(y_test, y_p)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['Kurang Menarik', 'Menarik']
    )
    disp.plot(ax=ax, colorbar=True, cmap=cmap)
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
    ax.set_xlabel('Label Prediksi', fontsize=10)
    ax.set_ylabel('Label Aktual',   fontsize=10)
    acc = accuracy_score(y_test, y_p)
    f1v = f1_score(y_test, y_p)
    ax.text(0.5, -0.22, f'Accuracy: {acc:.4f}  |  F1: {f1v:.4f}',
            ha='center', va='bottom', fontsize=9,
            transform=ax.transAxes, color='#333333')

plt.tight_layout()
plt.savefig('confusion_matrix_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ confusion_matrix_comparison.png disimpan')

---
## 🧪 Step 11 — Sanity Check Bisnis

Verifikasi logis: model harus memberikan **probabilitas lebih rendah**
untuk produk dengan harga 20× lebih mahal dari median pasarnya.
Jika tidak, ada sesuatu yang perlu diinvestigasi.

In [ ]:
contoh_df = df[df['label'] == 1].head(3)

for _, row in contoh_df.iterrows():
    kategori   = row['kategori']
    stat       = stat_per_kategori[stat_per_kategori['kategori'] == kategori].iloc[0]
    median_kat = stat['median_harga_kategori']

    def make_row(harga):
        rasio  = min(harga / max(median_kat, 1), 50)
        zscore = float(np.clip(
            (harga - stat['mean_harga_kategori']) / max(stat['std_harga_kategori'], 1),
            -5, 5
        ))
        log_h  = float(np.log1p(harga))
        q25, q75 = stat['q25_harga_kategori'], stat['q75_harga_kategori']
        seg    = 0 if harga <= q25 else (1 if harga <= q75 else 2)
        return pd.DataFrame([{
            'nama_produk_clean': row['nama_produk_clean'],
            'kategori'         : kategori,
            'sub_kategori'     : row['sub_kategori'],
            'rasio_harga'      : rasio,
            'zscore_harga'     : zscore,
            'log_harga'        : log_h,
            'segmen_harga'     : seg,
            'rating'           : row['rating'],
        }])

    prob_normal = best_rf.predict_proba(make_row(median_kat))[0][1]
    prob_gila   = best_rf.predict_proba(make_row(median_kat * 20))[0][1]

    status = '✅ TURUN' if prob_normal > prob_gila else '⚠️ TIDAK TURUN — investigasi!'
    print(f"Produk   : {row['nama_produk'][:55]}")
    print(f"Kategori : {kategori}  |  Median pasar: Rp{median_kat:,.0f}")
    print(f"  Harga wajar    (1× = Rp{median_kat:>12,.0f}): {prob_normal*100:.1f}% peluang laku")
    print(f"  Harga tdk wajar(20×= Rp{median_kat*20:>12,.0f}): {prob_gila*100:.1f}% peluang laku")
    print(f"  Δ Probabilitas : {(prob_normal-prob_gila)*100:.1f} poin  {status}")
    print()

---
## 💾 Step 12 — Retrain dengan Seluruh Data & Simpan Model Final

Model final dilatih ulang menggunakan **seluruh data** (train+val+test)
dengan parameter terbaik dari GridSearchCV, lalu disimpan ke `model_umkm_bogor_v2.joblib`.

In [ ]:
best_params = grid_search.best_params_

final_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators    =best_params['classifier__n_estimators'],
        max_depth       =best_params['classifier__max_depth'],
        min_samples_split=best_params['classifier__min_samples_split'],
        min_samples_leaf =best_params['classifier__min_samples_leaf'],
        class_weight    =best_params['classifier__class_weight'],
        random_state=42
    ))
])

print('Melatih model final dengan seluruh data...')
final_model.fit(X, y)

joblib.dump(final_model, 'model_umkm_bogor_v2.joblib')
print('\n✅ Model berhasil disimpan ke: model_umkm_bogor_v2.joblib')
print(f'   Best params: {best_params}')

---
## 📥 Step 13 — Download File Output dari Colab

Jalankan cell ini setelah semua step selesai.
Download **5 file** berikut dan taruh di folder `flask_api/` di laptop kamu:

| File | Dipakai oleh |
|---|---|
| `model_umkm_bogor_v2.joblib` | `app.py` — model ML utama |
| `market_stats_per_kategori.csv` | `app.py` — statistik harga pasar |
| `tren_per_kategori.csv` | `app.py` — tren antar kategori |
| `tren_per_sub_kategori.csv` | `app.py` — tren antar sub-kategori |
| `confusion_matrix_comparison.png` | Dokumentasi / laporan |

In [ ]:
import os
from google.colab import files

output_files = [
    'model_umkm_bogor_v2.joblib',
    'market_stats_per_kategori.csv',
    'tren_per_kategori.csv',
    'tren_per_sub_kategori.csv',
    'confusion_matrix_comparison.png',
]

print('Mengecek file yang tersedia...')
for fname in output_files:
    exists = os.path.exists(fname)
    size   = os.path.getsize(fname) if exists else 0
    status = f'✅  {size/1024:.1f} KB' if exists else '❌  tidak ditemukan'
    print(f'  {fname:<45} {status}')

print('\nMemulai download...')
for fname in output_files:
    if os.path.exists(fname):
        files.download(fname)
        print(f'  ⬇️  {fname}')
    else:
        print(f'  ⚠️  {fname} tidak ada, skip')

print('\n✅ Semua file selesai di-download!')
print('   Taruh semua file di folder: flask_api/')